In [1]:
import pandas as pd
from datetime import datetime, timezone, timedelta
import csv
import os
import requests
import json
import time
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

In [ ]:
# pushpull reddit search api로 순회 설정
# Reddit 전체는 불가능하다!! 반드시 subreddit 단위로 접근
# subreddit 설정 필요
SUBREDDITS = [
    "leagueoflegends",
    "summonerschool",
    "CompetitiveLoL"
]

START_DATE = datetime(2025, 1, 8, 0, 0, 0, tzinfo=timezone.utc)
END_DATE   = datetime(2026, 2, 4, 23, 59, 59, tzinfo=timezone.utc)

START_TS = int(START_DATE.timestamp())
END_TS   = int(END_DATE.timestamp())

OUTPUT_FILE = "lol_reddit_full_posts_comments.csv"

BASE_SUBMISSION_URL = "https://api.pullpush.io/reddit/search/submission/"
BASE_COMMENT_URL    = "https://api.pullpush.io/reddit/search/comment/"

BATCH_SIZE = 500
MAX_WORKERS = 6
MAX_RETRIES = 3

lock = threading.Lock()

In [4]:
# 중복 방지
saved_ids = set()

if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            saved_ids.add(row["post_id"])

print(f"이미 저장된 게시글 수: {len(saved_ids)}")

이미 저장된 게시글 수: 98


In [5]:
# csv 초기화
# 게시글 고유 id, 서브레딧 이름, 작성한 시간, 점수, 댓글 개수, 텍스트 full(제목, 본문, 댓글) <- 전처리 필요 
if not os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "post_id",
            "created_utc",
            "subreddit",
            "title",
            "selftext",
            "comment_id",
            "comment_body"
        ])

In [ ]:
# 하나의 게시글(post_id)에 달린 모든 댓글을 수집하는 함수
# PullPush는 한 번에 최대 100개만 반환하니까 파라미터를 갱신하면서 반복 호출 필요
def fetch_comments(post_id):
    comments = []
    current_after = 0
    while True:
        params = {"link_id": post_id, "after": current_after, "size": 500}
        retry_count = 0
        while retry_count < MAX_RETRIES:
            try:
                res = requests.get(BASE_COMMENT_URL, params=params, timeout=30)
                data = res.json().get("data", [])
                break
            except:
                retry_count += 1
                time.sleep(1)
        else:
            break
        if not data:
            break
        comments.extend(data)
        last_ts = data[-1]["created_utc"]
        if last_ts == current_after:
            break
        current_after = last_ts
    return comments

In [ ]:
# 게시글 + 댓글 처리
def process_post(post, subreddit):
    post_id = post["id"]
    created_ts = post["created_utc"]
    created_datetime_str = datetime.fromtimestamp(created_ts, timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

    comments = fetch_comments(post_id)
    rows = []

    if not comments:
        rows.append([
            post_id,
            created_ts,
            created_datetime_str,
            subreddit,
            post.get("score"),
            post.get("num_comments"),
            post.get("title"),
            post.get("selftext"),
            "",
            ""
        ])
    else:
        for c in comments:
            rows.append([
                post_id,
                created_ts,
                created_datetime_str,
                subreddit,
                post.get("score"),
                post.get("num_comments"),
                post.get("title"),
                post.get("selftext"),
                c.get("id"),
                c.get("body")
            ])
    return rows

In [ ]:
# 실제 존재하는 데이터 범위 탐색
def get_first_post_ts(subreddit):
    params = {
        "subreddit": subreddit,
        "sort": "asc",
        "sort_type": "created_utc",
        "size": 1
    }
    retry_count = 0
    while retry_count < MAX_RETRIES:
        try:
            res = requests.get(BASE_SUBMISSION_URL, params=params, timeout=30)
            data = res.json().get("data", [])
            if data:
                return max(data[0]["created_utc"], START_TS)
            else:
                return START_TS
        except:
            retry_count += 1
            time.sleep(1)
    return START_TS

In [ ]:
# 메인 실행
def collect():
    total_processed = 0
    start_time = time.time()

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "post_id",
            "created_utc",
            "created_datetime_utc",
            "subreddit",
            "score",
            "num_comments",
            "title",
            "selftext",
            "comment_id",
            "comment_body"
        ])

        for subreddit in SUBREDDITS:
            print(f"\n===== r/{subreddit} 수집 시작 =====")

            # 실제 존재하는 데이터가 있는 timestamp부터 시작
            current_after = get_first_post_ts(subreddit)

            while current_after < END_TS:
                params = {
                    "subreddit": subreddit,
                    "after": current_after,
                    "before": END_TS,
                    "size": BATCH_SIZE,
                    "sort": "asc",
                    "sort_type": "created_utc"
                }

                retry_count = 0
                while retry_count < MAX_RETRIES:
                    try:
                        res = requests.get(BASE_SUBMISSION_URL, params=params, timeout=30)
                        posts = res.json().get("data", [])
                        break
                    except Exception as e:
                        retry_count += 1
                        print(f"[r/{subreddit}] 요청 실패 ({retry_count}/3): {e}")
                        time.sleep(1)
                else:
                    print(f"[r/{subreddit}] 3회 실패, 다음 subreddit 이동")
                    break

                # START~END 범위 게시글만 필터
                posts = [p for p in posts if START_TS <= p["created_utc"] <= END_TS]

                if not posts:
                    # 데이터 없는 경우 → 다음 batch로 이동
                    current_after += BATCH_SIZE  # 그냥 다음 timestamp 블록으로 이동
                    continue  # 종료하지 않고 반복

                # 게시글 batch 병렬 처리
                with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                    futures = [executor.submit(process_post, post, subreddit) for post in posts]

                    for future in as_completed(futures):
                        rows = future.result()
                        with lock:
                            for row in rows:
                                writer.writerow(row)
                            total_processed += 1

                            created_dt_str = rows[0][2]
                            time_progress = (rows[0][1] - START_TS) / (END_TS - START_TS)
                            time_progress_percent = max(0, min(time_progress, 1)) * 100

                            elapsed_time = time.time() - start_time
                            avg_time = elapsed_time / max(total_processed,1)
                            remaining_time_est = avg_time * (len(posts) - total_processed)

                            if total_processed % 10 == 0:
                                print(
                                    f"[r/{subreddit}] 저장 중 ID: {rows[0][0]} | "
                                    f"게시글 UTC: {created_dt_str} | "
                                    f"범위 진행률: {time_progress_percent:.2f}% | "
                                    f"총 처리: {total_processed}개 | "
                                    f"평균: {avg_time:.2f}초/개 | "
                                    f"예상 남은 시간: {remaining_time_est/60:.1f}분"
                                )

                # 다음 batch 시작 timestamp = 마지막 게시글 timestamp + 1
                current_after = posts[-1]["created_utc"] + 1

    print("\n===== 전체 수집 완료 =====")


In [ ]:
# 실행
collect()

In [ ]:
# 필요하다면 진행 정글, 서폿 관련 용어 필터링 작업

df = pd.read_csv("lol_reddit_full_2025_2026.csv")

keywords = ["jungle", "jungler", "support", "roam", "gank"]
df_filtered = df[df["full_text"].str.contains("|".join(keywords), case=False)]
df_filtered.to_csv("lol_jungle_support_filtered.csv", index=False)